In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
!pip -q install -U transformers datasets peft accelerate scikit-learn pandas pyarrow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 63.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 89.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 72.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 12.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.2 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.2 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.2 w

In [5]:
%%writefile fedavg_tinybert_lora_sst2.py
import os
import re
import gc
import json
import time
import copy
import random
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Subset
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score

from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup,
    set_seed,
)
from peft import (
    LoraConfig,
    get_peft_model,
    PeftModel,
)
from transformers.utils import logging as hf_logging

hf_logging.set_verbosity_error()

# =========================================================
# CONFIG
# =========================================================
MODEL_NAME = "huawei-noah/TinyBERT_General_4L_312D"
DATASET_NAME = "sst2"
SETTING_NAME = "fedavg_tinybert_lora"
NUM_LABELS = 2

OUTPUT_DIR = "/content/drive/MyDrive/fedavg_tinybert_lora_sst2"
CHECKPOINTS_DIR = os.path.join(OUTPUT_DIR, "checkpoints")
BEST_MODEL_DIR = os.path.join(OUTPUT_DIR, "best_model")
FINAL_MODEL_DIR = os.path.join(OUTPUT_DIR, "final_model")

HISTORY_JSON = os.path.join(OUTPUT_DIR, "history.json")
HISTORY_CSV = os.path.join(OUTPUT_DIR, "history.csv")
CLIENT_HISTORY_JSON = os.path.join(OUTPUT_DIR, "client_history.json")
CLIENT_HISTORY_CSV = os.path.join(OUTPUT_DIR, "client_history.csv")
RUN_SUMMARY_JSON = os.path.join(OUTPUT_DIR, "run_summary.json")
CONFIG_JSON = os.path.join(OUTPUT_DIR, "config.json")

SEED = 42
MAX_LENGTH = 128

NUM_CLIENTS = 5
DIRICHLET_ALPHA = 0.5
PARTITION_TYPE = f"dirichlet_noniid_alpha_{DIRICHLET_ALPHA}"

LOCAL_EPOCHS = 1
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 64
LR = 2e-4
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0

LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.1
LORA_TARGET_MODULES = ["query", "key", "value"]

EARLY_STOPPING_PATIENCE = 3
MONITOR_METRIC = "accuracy"

# chỉ là trần an toàn; thực tế dừng bằng early stopping
MAX_ROUNDS = 1000000

NUM_WORKERS = 0
USE_FP16 = True

# =========================================================
# UTILS
# =========================================================
def ensure_dir(path):
    Path(path).mkdir(parents=True, exist_ok=True)

def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

def save_csv(rows, path):
    if len(rows) == 0:
        pd.DataFrame([]).to_csv(path, index=False)
    else:
        pd.DataFrame(rows).to_csv(path, index=False)

def save_histories(server_history, client_history):
    save_json(server_history, HISTORY_JSON)
    save_csv(server_history, HISTORY_CSV)
    save_json(client_history, CLIENT_HISTORY_JSON)
    save_csv(client_history, CLIENT_HISTORY_CSV)

def count_parameters(model):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return trainable_params, total_params

def state_dict_size_mb(state_dict):
    total_bytes = 0
    for _, v in state_dict.items():
        total_bytes += v.numel() * v.element_size()
    return total_bytes / (1024 ** 2)

def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

def tokenize_function(batch, tokenizer, max_length):
    return tokenizer(
        batch["sentence"],
        truncation=True,
        max_length=max_length,
    )

def dirichlet_partition(labels, num_clients=5, alpha=0.5, seed=42, min_size=10):
    rng = np.random.default_rng(seed)
    labels = np.array(labels)
    n_classes = len(np.unique(labels))

    while True:
        client_indices = [[] for _ in range(num_clients)]
        for c in range(n_classes):
            idx_c = np.where(labels == c)[0]
            rng.shuffle(idx_c)
            proportions = rng.dirichlet(np.repeat(alpha, num_clients))
            split_points = (np.cumsum(proportions) * len(idx_c)).astype(int)[:-1]
            splits = np.split(idx_c, split_points)
            for cid, split in enumerate(splits):
                client_indices[cid].extend(split.tolist())

        sizes = [len(x) for x in client_indices]
        if min(sizes) >= min_size:
            break

    for cid in range(num_clients):
        rng.shuffle(client_indices[cid])

    return client_indices

def list_checkpoints(checkpoints_dir):
    ckpts = []
    if not os.path.exists(checkpoints_dir):
        return ckpts
    for name in os.listdir(checkpoints_dir):
        full = os.path.join(checkpoints_dir, name)
        if os.path.isdir(full) and name.startswith("checkpoint_round_"):
            m = re.search(r"checkpoint_round_(\d+)", name)
            if m:
                ckpts.append((int(m.group(1)), full))
    ckpts.sort(key=lambda x: x[0])
    return ckpts

def latest_checkpoint_path(checkpoints_dir):
    ckpts = list_checkpoints(checkpoints_dir)
    return ckpts[-1][1] if len(ckpts) > 0 else None

def keep_latest_k_checkpoints(checkpoints_dir, keep_k=2):
    ckpts = list_checkpoints(checkpoints_dir)
    while len(ckpts) > keep_k:
        _, old_path = ckpts.pop(0)
        shutil.rmtree(old_path, ignore_errors=True)

def clear_and_make_dir(path):
    if os.path.exists(path):
        shutil.rmtree(path)
    os.makedirs(path, exist_ok=True)

def improved(current, best):
    if best is None:
        return True
    return current > best

# =========================================================
# MODEL / LORA
# =========================================================
def build_base_model():
    config = AutoConfig.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, config=config)
    return model

def build_peft_model():
    base_model = build_base_model()
    peft_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        task_type="SEQ_CLS",
        target_modules=LORA_TARGET_MODULES,
    )
    model = get_peft_model(base_model, peft_config)
    return model

def get_trainable_state_dict(model):
    trainable_keys = {name for name, p in model.named_parameters() if p.requires_grad}
    full_state = model.state_dict()
    return {k: v.detach().cpu().clone() for k, v in full_state.items() if k in trainable_keys}

def set_trainable_state_dict(model, trainable_state):
    current_state = model.state_dict()
    for k, v in trainable_state.items():
        if k in current_state:
            current_state[k] = v.detach().cpu().clone()
    model.load_state_dict(current_state, strict=False)

# =========================================================
# EVAL
# =========================================================
@torch.no_grad()
def evaluate(model, dataloader, device, use_fp16=False):
    model.eval()
    total_loss = 0.0
    total_count = 0
    all_preds = []
    all_labels = []

    amp_enabled = bool(use_fp16 and device.type == "cuda")

    for batch in dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}

        if amp_enabled:
            with torch.amp.autocast("cuda"):
                outputs = model(**batch)
                loss = outputs.loss
                logits = outputs.logits
        else:
            outputs = model(**batch)
            loss = outputs.loss
            logits = outputs.logits

        bs = batch["labels"].size(0)
        total_loss += loss.item() * bs
        total_count += bs

        preds = torch.argmax(logits, dim=-1)
        all_preds.extend(preds.detach().cpu().numpy().tolist())
        all_labels.extend(batch["labels"].detach().cpu().numpy().tolist())

    avg_loss = total_loss / max(total_count, 1)
    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")

    return {
        "eval_loss": float(avg_loss),
        "accuracy": float(acc),
        "macro_f1": float(macro_f1),
    }

# =========================================================
# FEDAVG ON TRAINABLE WEIGHTS
# =========================================================
def fedavg_aggregate_trainable_states(client_states, client_num_samples):
    total_samples = float(sum(client_num_samples))
    if total_samples <= 0:
        raise ValueError("Total client samples must be > 0")

    avg_state = {}
    keys = client_states[0].keys()

    for k in keys:
        avg_tensor = None
        for state, n in zip(client_states, client_num_samples):
            t = state[k].detach().float().cpu() * (n / total_samples)
            avg_tensor = t if avg_tensor is None else avg_tensor + t
        avg_state[k] = avg_tensor

    return avg_state

def train_one_client(
    client_id,
    client_model,
    global_trainable_state,
    train_loader,
    eval_loader,
    device,
    lr,
    local_epochs,
    weight_decay,
    round_idx,
    model_name,
    dataset_name,
    num_clients,
    partition_type,
    adapter_mb,
    trainable_params,
    total_params,
):
    set_trainable_state_dict(client_model, global_trainable_state)
    client_model.to(device)
    client_model.train()

    optimizer = torch.optim.AdamW(
        [p for p in client_model.parameters() if p.requires_grad],
        lr=lr,
        weight_decay=weight_decay,
    )

    total_steps = max(len(train_loader) * local_epochs, 1)
    warmup_steps = int(0.1 * total_steps)

    scheduler = get_linear_schedule_with_warmup(
        optimizer=optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )

    use_amp = bool(USE_FP16 and device.type == "cuda")
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp) if device.type == "cuda" else None

    start_time = time.time()
    running_loss = 0.0
    total_seen = 0

    for _ in range(local_epochs):
        client_model.train()

        for batch in train_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            optimizer.zero_grad(set_to_none=True)

            if use_amp:
                with torch.amp.autocast("cuda"):
                    outputs = client_model(**batch)
                    loss = outputs.loss
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(client_model.parameters(), MAX_GRAD_NORM)
                scaler.step(optimizer)
                scaler.update()
            else:
                outputs = client_model(**batch)
                loss = outputs.loss
                loss.backward()
                torch.nn.utils.clip_grad_norm_(client_model.parameters(), MAX_GRAD_NORM)
                optimizer.step()

            scheduler.step()

            bs = batch["labels"].size(0)
            running_loss += loss.item() * bs
            total_seen += bs

    train_time = time.time() - start_time
    train_loss = running_loss / max(total_seen, 1)

    client_eval = evaluate(client_model, eval_loader, device=device, use_fp16=USE_FP16)
    client_trainable_state = get_trainable_state_dict(client_model)

    client_log = {
        "round": int(round_idx),
        "accuracy": float(client_eval["accuracy"]),
        "macro_f1": float(client_eval["macro_f1"]),
        "eval_loss": float(client_eval["eval_loss"]),
        "round_time": None,
        "client_train_time": float(train_time),
        "communication_cost_MB": None,
        "trainable_params": int(trainable_params),
        "total_params": int(total_params),
        "model_name": str(model_name),
        "dataset_name": str(dataset_name),
        "setting": str(SETTING_NAME),
        "num_clients": int(num_clients),
        "local_epochs": int(local_epochs),
        "partition_type": str(partition_type),
        "best_metric_so_far": None,
        "patience_counter": None,
        "is_new_best": None,
        "client_id": int(client_id),
        "num_samples": int(total_seen),
        "train_loss": float(train_loss),
    }

    del optimizer, scheduler
    if scaler is not None:
        del scaler
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()

    return client_trainable_state, total_seen, train_time, client_log

# =========================================================
# SAVE / LOAD
# =========================================================
def save_model_bundle(save_dir, model, tokenizer):
    clear_and_make_dir(save_dir)
    model.save_pretrained(save_dir)
    tokenizer.save_pretrained(save_dir)
    metadata = {
        "base_model_name": MODEL_NAME,
        "dataset_name": DATASET_NAME,
        "setting": SETTING_NAME,
        "num_labels": NUM_LABELS,
        "lora_r": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "lora_dropout": LORA_DROPOUT,
        "lora_target_modules": LORA_TARGET_MODULES,
    }
    save_json(metadata, os.path.join(save_dir, "model_metadata.json"))

def save_checkpoint(
    round_idx,
    global_model,
    tokenizer,
    best_metric_so_far,
    patience_counter,
    server_history,
    client_history,
):
    ckpt_dir = os.path.join(CHECKPOINTS_DIR, f"checkpoint_round_{round_idx:06d}")
    ensure_dir(ckpt_dir)

    global_model.save_pretrained(os.path.join(ckpt_dir, "model"))
    tokenizer.save_pretrained(os.path.join(ckpt_dir, "tokenizer"))

    state = {
        "round": int(round_idx),
        "best_metric_so_far": None if best_metric_so_far is None else float(best_metric_so_far),
        "patience_counter": int(patience_counter),
        "server_history": server_history,
        "client_history": client_history,
        "config": {
            "model_name": MODEL_NAME,
            "dataset_name": DATASET_NAME,
            "setting": SETTING_NAME,
            "output_dir": OUTPUT_DIR,
            "checkpoints_dir": CHECKPOINTS_DIR,
            "best_model_dir": BEST_MODEL_DIR,
            "final_model_dir": FINAL_MODEL_DIR,
            "history_json": HISTORY_JSON,
            "history_csv": HISTORY_CSV,
            "client_history_json": CLIENT_HISTORY_JSON,
            "client_history_csv": CLIENT_HISTORY_CSV,
            "num_clients": NUM_CLIENTS,
            "dirichlet_alpha": DIRICHLET_ALPHA,
            "partition_type": PARTITION_TYPE,
            "local_epochs": LOCAL_EPOCHS,
            "train_batch_size": TRAIN_BATCH_SIZE,
            "eval_batch_size": EVAL_BATCH_SIZE,
            "lr": LR,
            "weight_decay": WEIGHT_DECAY,
            "max_length": MAX_LENGTH,
            "early_stopping_patience": EARLY_STOPPING_PATIENCE,
            "seed": SEED,
            "lora_r": LORA_R,
            "lora_alpha": LORA_ALPHA,
            "lora_dropout": LORA_DROPOUT,
            "lora_target_modules": LORA_TARGET_MODULES,
        }
    }
    torch.save(state, os.path.join(ckpt_dir, "training_state.pt"))

    with open(os.path.join(CHECKPOINTS_DIR, "latest_checkpoint.txt"), "w", encoding="utf-8") as f:
        f.write(ckpt_dir)

    keep_latest_k_checkpoints(CHECKPOINTS_DIR, keep_k=2)
    return ckpt_dir

def load_latest_checkpoint(device):
    ckpt_path = latest_checkpoint_path(CHECKPOINTS_DIR)
    if ckpt_path is None:
        return None

    model_dir = os.path.join(ckpt_path, "model")
    tokenizer_dir = os.path.join(ckpt_path, "tokenizer")
    state_path = os.path.join(ckpt_path, "training_state.pt")

    if not (os.path.exists(model_dir) and os.path.exists(tokenizer_dir) and os.path.exists(state_path)):
        return None

    base_model = build_base_model()
    global_model = PeftModel.from_pretrained(base_model, model_dir)
    global_model.to(device)

    tokenizer = AutoTokenizer.from_pretrained(tokenizer_dir, use_fast=True)
    state = torch.load(state_path, map_location=device)

    return {
        "ckpt_path": ckpt_path,
        "global_model": global_model,
        "tokenizer": tokenizer,
        "start_round": int(state["round"]) + 1,
        "best_metric_so_far": state["best_metric_so_far"],
        "patience_counter": int(state["patience_counter"]),
        "server_history": state["server_history"],
        "client_history": state["client_history"],
    }

# =========================================================
# MAIN
# =========================================================
def main():
    ensure_dir(OUTPUT_DIR)
    ensure_dir(CHECKPOINTS_DIR)
    ensure_dir(BEST_MODEL_DIR)
    ensure_dir(FINAL_MODEL_DIR)

    config_dump = {
        "model_name": MODEL_NAME,
        "dataset_name": DATASET_NAME,
        "setting": SETTING_NAME,
        "output_dir": OUTPUT_DIR,
        "checkpoints_dir": CHECKPOINTS_DIR,
        "best_model_dir": BEST_MODEL_DIR,
        "final_model_dir": FINAL_MODEL_DIR,
        "history_json": HISTORY_JSON,
        "history_csv": HISTORY_CSV,
        "client_history_json": CLIENT_HISTORY_JSON,
        "client_history_csv": CLIENT_HISTORY_CSV,
        "run_summary_json": RUN_SUMMARY_JSON,
        "num_clients": NUM_CLIENTS,
        "dirichlet_alpha": DIRICHLET_ALPHA,
        "partition_type": PARTITION_TYPE,
        "local_epochs": LOCAL_EPOCHS,
        "train_batch_size": TRAIN_BATCH_SIZE,
        "eval_batch_size": EVAL_BATCH_SIZE,
        "lr": LR,
        "weight_decay": WEIGHT_DECAY,
        "max_length": MAX_LENGTH,
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "monitor_metric": MONITOR_METRIC,
        "seed": SEED,
        "lora_r": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "lora_dropout": LORA_DROPOUT,
        "lora_target_modules": LORA_TARGET_MODULES,
    }
    save_json(config_dump, CONFIG_JSON)

    set_seed(SEED)
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    device = get_device()

    print("=" * 120)
    print("Federated Learning: FedAvg + TinyBERT + LoRA on SST-2")
    print(f"Device: {device}")
    print(f"Output directory: {OUTPUT_DIR}")
    print(f"Checkpoints directory: {CHECKPOINTS_DIR}")
    print(f"Best model directory: {BEST_MODEL_DIR}")
    print(f"Final model directory: {FINAL_MODEL_DIR}")
    print("=" * 120)

    resume_info = load_latest_checkpoint(device=device)

    if resume_info is not None:
        print(f"[Resume] Found latest checkpoint: {resume_info['ckpt_path']}")
        tokenizer = resume_info["tokenizer"]
        global_model = resume_info["global_model"]
        start_round = resume_info["start_round"]
        best_metric_so_far = resume_info["best_metric_so_far"]
        patience_counter = resume_info["patience_counter"]
        server_history = resume_info["server_history"]
        client_history = resume_info["client_history"]
        print(f"[Resume] Continuing from round {start_round}")
    else:
        print("[Start] No checkpoint found. Starting from scratch.")
        tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
        global_model = build_peft_model().to(device)
        start_round = 1
        best_metric_so_far = None
        patience_counter = 0
        server_history = []
        client_history = []

    trainable_params, total_params = count_parameters(global_model)

    raw_datasets = load_dataset("glue", DATASET_NAME)

    tokenized_train = raw_datasets["train"].map(
        lambda batch: tokenize_function(batch, tokenizer, MAX_LENGTH),
        batched=True,
        remove_columns=["sentence", "idx"],
    )
    tokenized_val = raw_datasets["validation"].map(
        lambda batch: tokenize_function(batch, tokenizer, MAX_LENGTH),
        batched=True,
        remove_columns=["sentence", "idx"],
    )

    tokenized_train = tokenized_train.rename_column("label", "labels")
    tokenized_val = tokenized_val.rename_column("label", "labels")

    tokenized_train.set_format(type="torch")
    tokenized_val.set_format(type="torch")

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    val_loader = DataLoader(
        tokenized_val,
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False,
        collate_fn=data_collator,
        num_workers=NUM_WORKERS,
        pin_memory=(device.type == "cuda"),
    )

    train_labels = tokenized_train["labels"]
    client_indices = dirichlet_partition(
        labels=train_labels,
        num_clients=NUM_CLIENTS,
        alpha=DIRICHLET_ALPHA,
        seed=SEED,
        min_size=10,
    )

    print(f"Train samples: {len(tokenized_train)}")
    print(f"Validation samples: {len(tokenized_val)}")
    print(f"Model: {MODEL_NAME}")
    print(f"Trainable params: {trainable_params}")
    print(f"Total params: {total_params}")
    print(f"Number of clients: {NUM_CLIENTS}")
    print(f"Partition type: {PARTITION_TYPE}")
    for cid in range(NUM_CLIENTS):
        print(f" - Client {cid}: {len(client_indices[cid])} samples")

    stopped_early = False
    round_idx = start_round

    while True:
        print("\n" + "=" * 40 + f" Round {round_idx} " + "=" * 40, flush=True)
        round_start_time = time.time()

        global_trainable_state = get_trainable_state_dict(global_model)
        adapter_mb = state_dict_size_mb(global_trainable_state)

        client_states = []
        client_num_samples = []
        client_train_times = []
        round_client_logs = []

        # server broadcast + client local training tuần tự
        for client_id in range(NUM_CLIENTS):
            print(f"[Round {round_idx}] Start client {client_id}", flush=True)

            subset = Subset(tokenized_train, client_indices[client_id])
            train_loader = DataLoader(
                subset,
                batch_size=TRAIN_BATCH_SIZE,
                shuffle=True,
                collate_fn=data_collator,
                num_workers=NUM_WORKERS,
                pin_memory=(device.type == "cuda"),
            )

            client_model = build_peft_model().to(device)

            client_state, num_samples, train_time, client_log = train_one_client(
                client_id=client_id,
                client_model=client_model,
                global_trainable_state=global_trainable_state,
                train_loader=train_loader,
                eval_loader=val_loader,
                device=device,
                lr=LR,
                local_epochs=LOCAL_EPOCHS,
                weight_decay=WEIGHT_DECAY,
                round_idx=round_idx,
                model_name=MODEL_NAME,
                dataset_name=DATASET_NAME,
                num_clients=NUM_CLIENTS,
                partition_type=PARTITION_TYPE,
                adapter_mb=adapter_mb,
                trainable_params=trainable_params,
                total_params=total_params,
            )

            client_states.append(client_state)
            client_num_samples.append(num_samples)
            client_train_times.append(train_time)
            round_client_logs.append(client_log)

            print(
                f"[Round {round_idx}] Finish client {client_id} | "
                f"samples={num_samples} | train_loss={client_log['train_loss']:.6f}",
                flush=True
            )

            del client_model, subset, train_loader
            gc.collect()
            if device.type == "cuda":
                torch.cuda.empty_cache()

        # server aggregation
        aggregated_trainable_state = fedavg_aggregate_trainable_states(client_states, client_num_samples)
        set_trainable_state_dict(global_model, aggregated_trainable_state)
        global_model.to(device)

        # server evaluation
        server_eval = evaluate(global_model, val_loader, device=device, use_fp16=USE_FP16)

        round_time = time.time() - round_start_time
        client_train_time_total = float(sum(client_train_times))
        communication_cost_mb = float(adapter_mb * 2.0 * NUM_CLIENTS)

        current_metric = float(server_eval[MONITOR_METRIC])
        is_new_best = improved(current_metric, best_metric_so_far)

        if is_new_best:
            best_metric_so_far = current_metric
            patience_counter = 0
            save_model_bundle(BEST_MODEL_DIR, global_model, tokenizer)
        else:
            patience_counter += 1

        server_log = {
            "round": int(round_idx),
            "accuracy": float(server_eval["accuracy"]),
            "macro_f1": float(server_eval["macro_f1"]),
            "eval_loss": float(server_eval["eval_loss"]),
            "round_time": float(round_time),
            "client_train_time": float(client_train_time_total),
            "communication_cost_MB": float(communication_cost_mb),
            "trainable_params": int(trainable_params),
            "total_params": int(total_params),
            "model_name": str(MODEL_NAME),
            "dataset_name": str(DATASET_NAME),
            "setting": str(SETTING_NAME),
            "num_clients": int(NUM_CLIENTS),
            "local_epochs": int(LOCAL_EPOCHS),
            "partition_type": str(PARTITION_TYPE),
            "best_metric_so_far": float(best_metric_so_far),
            "patience_counter": int(patience_counter),
            "is_new_best": bool(is_new_best),
        }
        server_history.append(server_log)

        for client_log in round_client_logs:
            client_log["round_time"] = float(round_time)
            client_log["communication_cost_MB"] = float(communication_cost_mb)
            client_log["best_metric_so_far"] = float(best_metric_so_far)
            client_log["patience_counter"] = int(patience_counter)
            client_log["is_new_best"] = bool(is_new_best)
            client_history.append(client_log)

        save_histories(server_history, client_history)

        ckpt_dir = save_checkpoint(
            round_idx=round_idx,
            global_model=global_model,
            tokenizer=tokenizer,
            best_metric_so_far=best_metric_so_far,
            patience_counter=patience_counter,
            server_history=server_history,
            client_history=client_history,
        )

        print("-" * 120)
        print("SERVER ROUND LOG:")
        print(json.dumps(server_log, indent=2))
        print(f"Checkpoint saved: {ckpt_dir}")
        print("-" * 120)

        gc.collect()
        if device.type == "cuda":
            torch.cuda.empty_cache()

        if patience_counter >= EARLY_STOPPING_PATIENCE:
            print(f"[Early Stopping] No validation {MONITOR_METRIC} improvement for {EARLY_STOPPING_PATIENCE} consecutive rounds.")
            stopped_early = True
            break

        round_idx += 1

        if round_idx > MAX_ROUNDS:
            print("[Stop] Reached MAX_ROUNDS safety ceiling.")
            break

    save_model_bundle(FINAL_MODEL_DIR, global_model, tokenizer)
    save_histories(server_history, client_history)

    run_summary = {
        "model_name": MODEL_NAME,
        "dataset_name": DATASET_NAME,
        "setting": SETTING_NAME,
        "output_dir": OUTPUT_DIR,
        "checkpoints_dir": CHECKPOINTS_DIR,
        "best_model_dir": BEST_MODEL_DIR,
        "final_model_dir": FINAL_MODEL_DIR,
        "history_json": HISTORY_JSON,
        "history_csv": HISTORY_CSV,
        "client_history_json": CLIENT_HISTORY_JSON,
        "client_history_csv": CLIENT_HISTORY_CSV,
        "num_rounds_completed": int(len(server_history)),
        "num_client_logs": int(len(client_history)),
        "best_metric_so_far": None if best_metric_so_far is None else float(best_metric_so_far),
        "stopped_early": bool(stopped_early),
        "latest_checkpoint": latest_checkpoint_path(CHECKPOINTS_DIR),
    }
    save_json(run_summary, RUN_SUMMARY_JSON)

    # SELF-CHECK
    expected_server_cols = [
        "round",
        "accuracy",
        "macro_f1",
        "eval_loss",
        "round_time",
        "client_train_time",
        "communication_cost_MB",
        "trainable_params",
        "total_params",
        "model_name",
        "dataset_name",
        "setting",
        "num_clients",
        "local_epochs",
        "partition_type",
        "best_metric_so_far",
        "patience_counter",
        "is_new_best",
    ]

    expected_client_cols = expected_server_cols + [
        "client_id",
        "num_samples",
        "train_loss",
    ]

    server_df = pd.DataFrame(server_history)
    client_df = pd.DataFrame(client_history)

    missing_server_cols = [c for c in expected_server_cols if c not in server_df.columns]
    missing_client_cols = [c for c in expected_client_cols if c not in client_df.columns]

    required_paths = [
        OUTPUT_DIR,
        CHECKPOINTS_DIR,
        BEST_MODEL_DIR,
        FINAL_MODEL_DIR,
        HISTORY_JSON,
        HISTORY_CSV,
        CLIENT_HISTORY_JSON,
        CLIENT_HISTORY_CSV,
        RUN_SUMMARY_JSON,
        CONFIG_JSON,
    ]
    missing_files = [p for p in required_paths if not os.path.exists(p)]

    print("=" * 120)
    print("Training completed.")
    print(f"Output directory: {OUTPUT_DIR}")
    print("Generated files/folders:")
    print(f" - {HISTORY_JSON}")
    print(f" - {HISTORY_CSV}")
    print(f" - {CLIENT_HISTORY_JSON}")
    print(f" - {CLIENT_HISTORY_CSV}")
    print(f" - {RUN_SUMMARY_JSON}")
    print(f" - {CONFIG_JSON}")
    print(f" - {CHECKPOINTS_DIR}/latest_checkpoint.txt")
    print(f" - {CHECKPOINTS_DIR}/checkpoint_round_xxxxxx/")
    print(f" - {BEST_MODEL_DIR}/")
    print(f" - {FINAL_MODEL_DIR}/")
    print("-" * 120)
    print("[SELF-CHECK]")
    print(f"Missing server columns: {missing_server_cols}")
    print(f"Missing client columns: {missing_client_cols}")
    print(f"Missing files/folders: {missing_files}")
    print("=" * 120)

    if len(missing_server_cols) == 0 and len(missing_client_cols) == 0 and len(missing_files) == 0:
        print("[SELF-CHECK PASSED] All required columns and output files are present.")
    else:
        raise ValueError("SELF-CHECK FAILED: missing required columns or files.")

if __name__ == "__main__":
    main()

Overwriting fedavg_tinybert_lora_sst2.py


In [6]:
# Xóa output cũ nếu trước đó đã chạy lỗi
!rm -rf /content/drive/MyDrive/fedavg_tinybert_lora_sst2

!python fedavg_tinybert_lora_sst2.py

Federated Learning: FedAvg + TinyBERT + LoRA on SST-2
Device: cuda
Output directory: /content/drive/MyDrive/fedavg_tinybert_lora_sst2
Checkpoints directory: /content/drive/MyDrive/fedavg_tinybert_lora_sst2/checkpoints
Best model directory: /content/drive/MyDrive/fedavg_tinybert_lora_sst2/best_model
Final model directory: /content/drive/MyDrive/fedavg_tinybert_lora_sst2/final_model
[Start] No checkpoint found. Starting from scratch.
Loading weights: 100% 71/71 [00:00<00:00, 27550.71it/s]
Using the latest cached version of the dataset since glue couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'sst2' at /root/.cache/huggingface/datasets/glue/sst2/0.0.0/bcdcba79d07bc864c1c254ccfcedcce55bcc9a8c (last modified on Wed Apr  8 00:21:20 2026).
Train samples: 67349
Validation samples: 872
Model: huawei-noah/TinyBERT_General_4L_312D
Trainable params: 60530
Total params: 14411404
Number of clients: 5
Partition type: dirichlet_noniid_alpha_0.5
 - Client 0: 562